In [1]:
import os
import pandas as pd
import requests_cache
from retry_requests import retry
import openmeteo_requests

In [2]:
# ---------- Setup API Client ----------
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

In [3]:
# ---------- Cities with Coordinates ----------
CITIES = {
    "Athens": (37.98, 23.72),
    "London": (51.51, -0.13),
    "Paris": (48.85, 2.35),
    "Berlin": (52.52, 13.41),
    "Madrid": (40.42, -3.70),
    "Rome": (41.90, 12.50),
    "Lisbon": (38.72, -9.14),
    "Vienna": (48.21, 16.37),
    "Warsaw": (52.23, 21.01),
    "Prague": (50.08, 14.43),
    "Budapest": (47.50, 19.04),
    "Amsterdam": (52.37, 4.90),
    "Brussels": (50.85, 4.35),
    "Copenhagen": (55.68, 12.57),
    "Stockholm": (59.33, 18.07),
    "Oslo": (59.91, 10.75),
    "Helsinki": (60.17, 24.94),
    "Dublin": (53.33, -6.25),
    "Zurich": (47.37, 8.55),
    "Istanbul": (41.01, 28.95),
    "Moscow": (55.75, 37.62),
    "New York": (40.71, -74.01),
    "Los Angeles": (34.05, -118.24),
    "Chicago": (41.88, -87.63),
    "Toronto": (43.65, -79.38),
    "Vancouver": (49.28, -123.12),
    "Mexico City": (19.43, -99.13),
    "São Paulo": (-23.55, -46.63),
    "Buenos Aires": (-34.61, -58.38),
    "Santiago": (-33.45, -70.67),
    "Cape Town": (-33.93, 18.42),
    "Cairo": (30.04, 31.24),
    "Nairobi": (-1.29, 36.82),
    "Dubai": (25.20, 55.27),
    "Mumbai": (19.08, 72.88),
    "Delhi": (28.61, 77.21),
    "Beijing": (39.91, 116.40),
    "Shanghai": (31.23, 121.47),
    "Tokyo": (35.68, 139.76),
    "Seoul": (37.57, 126.98),
    "Bangkok": (13.75, 100.50),
    "Singapore": (1.29, 103.85),
    "Sydney": (-33.87, 151.21),
    "Melbourne": (-37.81, 144.96),
    "Auckland": (-36.85, 174.76),
    "Hong Kong": (22.32, 114.17),
    "Kuala Lumpur": (3.14, 101.69),
    "Jakarta": (-6.21, 106.85),
    "Manila": (14.60, 120.98)
}

In [4]:
# ---------- Weather Fetch Function ----------
def fetch_weather(city: str, lat: float, lon: float, days: int = 60) -> pd.DataFrame:
    """Fetch past weather + forecast data for a city."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "rain"],
        "past_days": days,
        "forecast_days": 1,
    }
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]

    hourly = response.Hourly()
    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        ),
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
        "wind_speed_10m": hourly.Variables(2).ValuesAsNumpy(),
        "rain": hourly.Variables(3).ValuesAsNumpy(),
    }

    return pd.DataFrame(hourly_data)

In [5]:
# ---------- Save Data ----------
def save_weather_data(city: str, df: pd.DataFrame):
    os.makedirs("../data", exist_ok=True)
    file_path = f"../data/weather_{city.replace(' ', '_')}.csv"
    df.to_csv(file_path, index=False)
    print(f"✅ Saved {city} data to {file_path}")

In [6]:
if __name__ == "__main__":
    # Example: Choose a city from the dictionary
    city = "Madrid"
    lat, lon = CITIES[city]

    df = fetch_weather(city, lat, lon, days=60)
    save_weather_data(city, df)

    print(df.head())

✅ Saved Madrid data to ../data/weather_Madrid.csv
                       date  temperature_2m  relative_humidity_2m  \
0 2025-07-26 00:00:00+00:00       20.859001                  41.0   
1 2025-07-26 01:00:00+00:00       19.509001                  45.0   
2 2025-07-26 02:00:00+00:00       18.459000                  48.0   
3 2025-07-26 03:00:00+00:00       17.409000                  54.0   
4 2025-07-26 04:00:00+00:00       16.709000                  56.0   

   wind_speed_10m  rain  
0       12.738099   0.0  
1       11.246759   0.0  
2       11.592894   0.0  
3       11.090103   0.0  
4       10.464797   0.0  


## Data Processing

In [7]:
df['date'] = pd.to_datetime(df['date'])

# Find the last date (without time)
last_day = df['date'].dt.date.max()

# Filter out rows from the last day
df = df[df['date'].dt.date != last_day]

In [8]:
df.tail()

,date,temperature_2m,relative_humidity_2m,wind_speed_10m,rain
1435,2025-09-23 19:00:00+00:00,18.209000,30.0,11.720751,0.0
1436,2025-09-23 20:00:00+00:00,16.909000,32.0,12.758432,0.0
1437,2025-09-23 21:00:00+00:00,15.759001,34.0,11.966953,0.0
1438,2025-09-23 22:00:00+00:00,14.809000,37.0,10.464797,0.0
1439,2025-09-23 23:00:00+00:00,13.809000,42.0,12.481153,0.0
